# Gene-based testing and a binary-trait exome study

**Purpose.** A single-variant GWAS has almost no power for **rare** variants: each one is
seen in too few people. Gene-based tests get around this by asking whether the rare variants
in a *gene*, taken together, associate with the trait. This exercise runs those tests on
whole-exome data for a binary disease trait, which is where they matter most.

**What you will do**
 - fit a null linear mixed model with `regenie` for a **binary** trait, using approximate
   Firth correction for the case/control imbalance
 - run single-variant association on the exome data, chromosome by chromosome
 - build the three inputs gene-based testing needs — an **annotation** file, a **set list**
   of the variants in each gene, and a **mask** file defining which variant classes to
   include
 - run **SKAT** and **ACAT** and read the gene-level results
 - calculate power for quantitative and binary traits

**The data.** Whole-exome sequencing for **Charcot-Marie-Tooth disease**, a binary
case/control trait, with a covariate file for 1,826 individuals, plus the annotation, set
list and mask files that define the gene-based tests. **Called genotypes** from exome
sequencing.

> ## ⚠️ The data for this exercise is not on this server
>
> Like the morning GWAS exercise, this notebook expects its files under
> `/home/student/<username>/GWAS/data/`, a per-student folder on the course machine. That
> folder does not exist here and the dataset is in none of the course archives. The
> exercise is included **unchanged** so the material is not lost; all the paths are in the
> setup cell, so pointing `DATA` at the files is the only edit needed once they are found.

**Relationship to the morning exercise.** This notebook repeats Exercises D, E and F from
[the full GWAS analysis](gwas_analysis_human.ipynb) — 45 of its 54 cells are the same
commands. What is genuinely different is the **data and the trait**: the morning runs on
array genotypes with a quantitative phenotype, this one on **exome data with a binary
disease trait**, which changes how the null model is fitted (Firth correction) and is the
setting where gene-based tests are actually used. Do the morning one first.

## Setup

All the paths used by this exercise are set in the cell below.

In [ ]:
#############################################################
# ALL PATHS ARE SET HERE
# If the data moves, this is the ONLY cell you need to change.
# No cell below this one uses a full path.
#############################################################

# Where the exome data lives.
#
# NOTE: this dataset is NOT currently on this server. It was written for a
# course machine where every student had a personal copy under
#     /home/student/<username>/GWAS/data/
DATA=$HOME/GWAS/data

# the conda environment the course machine used, also not present here
CONDA_ACTIVATE=$HOME/miniconda3/bin/activate
CONDA_ENV=regenie_env

# where you will do the exercise
WORK_DIR=$HOME/gene_based_testing_human
mkdir -p $WORK_DIR/output
echo $WORK_DIR > $HOME/.gene_based_workdir
cd $WORK_DIR

if [ -d "$DATA" ]; then
    echo "data folder: $DATA"; ls $DATA | head
else
    echo "NOTE: the data folder $DATA was not found."
    echo "This exercise needs the exome dataset, which is not on this server yet."
fi

# Exercise D: linear mixed model

## Part 2: Binary traits

**Charcot-Marie-Tooth Disease (CMT)** is a group of inherited peripheral neuropathies characterized by progressive muscle weakness, atrophy, and sensory loss, primarily in the distal limbs. It is one of the most common inherited neurological disorders, affecting approximately 1 in 2,500 people. Symptoms often begin in adolescence or early adulthood and may include foot deformities, gait abnormalities, and reduced reflexes.

Below you will apply a method called [regenie](https://rgcgithub.github.io/regenie/) that implemented linear mixed model for GWAS.



In [ ]:
# create the output folder for storing output files
if [ ! -d ./output ]; then
  mkdir ./output
fi

## Step0: Format the data as needed


In [ ]:
head $DATA/covar_2024_WES_1826_phenotype.txt

**Question**
 - The covariate file has 1,826 individuals. Which columns are covariates and which is the phenotype, and why is the trait coded 0/1 here rather than as a measurement?

## Step1: fitting the null linear mixed model with regenie

For quantitative traits (such as standing height), regenie fits a linear mixed model by default.  
It is recommended to inverse normalize the phenotype before analysis to improve normality and statistical power.

In [ ]:
# # do not run!
# # running about 2.5 hours
# source $CONDA_ACTIVATE $CONDA_ENV
# regenie \
#   --step 1 \
#   --bed $DATA/ukb23150_c1_c22_2024_WES_1826_QCpruned \
#   --phenoFile $DATA/covar_2024_WES_1826_phenotype.txt \
#   --bt \
#   --strict \
#   --firth --approx \
#   --bsize 1000 \
#   --loocv \
#   --lowmem \
#   --lowmem-prefix ./output/regenie_tmp_preds_WES \
#   --write-null-firth \
#   --out ./output/regenie_step1_WO_PC20_WES
# conda deactivate

**Questions**
 - Step 1 fits a null model with no single variant in it. What is it estimating, and why must it come first?
 - `--firth` is used here but not for the quantitative trait. What problem is it correcting for?

In [ ]:
# files .loco is the output with the per-chromosome LOCO predictions as rows of the files
head -n 22 $DATA/regenie_step1_WES_1.loco |
  awk '{print $1,$2,$3,$4,$5,$6,$7,$8,$9}' | column -t

**Question**
 - The `.loco` file holds one prediction per chromosome. Why leave out the chromosome being tested rather than using a single genome-wide prediction?

See more parameter explanations: https://rgcgithub.github.io/regenie/options/.

The estimates for approximate Firth under the null will be written to file .firth and the list of these files is written to file_firth.list. This can be used in step 2 as --use-null-firth file_firth.list.

In [ ]:
cat $DATA/regenie_step1_WES_1.firth

**Question**
 - The `.firth` file records where the Firth correction was applied. Which variants would you expect to need it most?

## Step 2: performing single-variant association tests

For quantitative traits (such as standing height), regenie automatically uses a linear mixed model for association testing.

There is no need for saddle point approximation or Firth correction, as these are specific to binary traits.

The output will include effect sizes, standard errors, and p-values for each variant.

In [ ]:
# running about 0.5 minutes
source $CONDA_ACTIVATE $CONDA_ENV
regenie \
  --step 2 \
  --bed $DATA/ukb23150_c1_c22_2024_WES_1826_QCpruned \
  --ref-first \
  --phenoFile $DATA/covar_2024_WES_1826_phenotype.txt \
  --bt \
  --strict \
  --bsize 1000 \
  --firth --approx \
  --pThresh 0.01 \
  --pred $DATA/regenie_step1_WES_pred.list \
  --use-null-firth $DATA/regenie_step1_WES_firth.list \
  --out ./output/regenie_step2_asso_WES
conda deactivate

**Question**
 - Step 2 tests variants one at a time, conditioning on the step-1 predictions. What would happen to the p-values without that conditioning?

### View the result of REGENIE-GWAS


In [ ]:
head ./output/regenie_step2_asso_WES_phenotype.regenie | column -t

**Question**
 - Read the output columns. Which holds the test statistic and which the p-value, and what is `LOG10P`?

In [ ]:
# search for the most significant SNP
awk 'NR>1 {print $1,$2,$3,$12}' \
  ./output/regenie_step2_asso_WES_phenotype.regenie | sort -k4,4gr 2>/dev/null | head -1 || true

**Question**
 - What is the most significant single variant, and is it significant after correcting for the number tested?

In [ ]:
# Prepare for plotting the results
options(warn = -1)
suppressPackageStartupMessages({
  library(magrittr)
})
regenie_results= data.table::fread("./output/regenie_step2_asso_WES_phenotype.regenie")
regenie_results[, P := 10^(-LOG10P)] %>%
    data.table::setnames(
        c("CHROM", "GENPOS", "ID"),
        c("CHR", "BP", "SNP")
    )
regenie_results %>%
    data.table::fwrite("./output/regenie_step2_asso_WES_phenotype_v2.regenie")

In [ ]:
# running about 0.5 minutes
options(repr.plot.width = 16,
        repr.plot.height = 4,
        repr.plot.res = 600)
source("$DATA/plotPlink.R")

# plot the results (.regenie is the output file from regenie)
plots= plot_qqman(
  plink_assoc_file= "./output/regenie_step2_asso_WES_phenotype_v2.regenie",
  pheno_name= "CMT",
  save_plot = FALSE,
  lambda1_qq_pos = c(1.48, -5.5),
  lambda2_qq_pos = c(1.1, -4)
)
print(plots$manhattan_plot)

**Questions**
 - Does the Manhattan plot show any peak?
 - For a rare-variant exome study, would you expect one from single-variant tests?

In [ ]:
# running about 0.5 minutes
options(repr.plot.width = 4.1,
        repr.plot.height = 4.1,
        repr.plot.res = 600)
print(plots$qq_plot)

**Question**
 - Look at the QQ plot. Does it follow the diagonal, and what would a systematic departure mean here?

# Exercise E : Gene-based testing

Instead of performing single-variant association tests, multiple variants can be aggregated in a given region, such as a gene.

This can be especially helpful when testing **rare variants** as single-vatiant tests usuaally have lower power performance.

To avoid inflation in the gene-based tets due to rare variants as well as reduce computation time, we can implement the collapsing approach of gene-based testing proposed in SAIGE-GENE+, where ultra-rare variants are aggregated into a mask.

### Annotation input files: to define variant sets and functional annotations which will be used to generate masks.

Each line contains the variant name, the set/gene name and a single annotation category (space/tab separated).

Variants not in this file will be assigned to a default "NULL" category. A maximum of 63 annotation categories (+NULL category) is allowed.

To obtain a single annotation per gene, we could choose the most deleterious functional annotation across the gene transcripts or alternatively use the canonical transcript (note that its definition can vary across software).

In [ ]:
head $DATA/regenie_WES_anno_file.txt | column -t

**Question**
 - The annotation file assigns each variant to a gene and a functional class. Why does a gene-based test need the functional class as well as the gene?

### Set list file: to list variants within each set/gene to use when building masks.

Each line contains the set/gene name followed by a chromosome and physical position for the set/gene, then by a comma-separated list of variants included in the set/gene.


In [ ]:
head -n 1 $DATA/regenie_WES_set_list_CHR_POS.txt | column -t

**Question**
 - The set list names the variants in each gene. How many variants does a typical gene contribute, and what does that say about the power of a single-variant test?

### Mask file

This file specifies which annotation categories should be combined into masks.

Each line contains a mask name followed by a comma-separated list of categories included in the mask (i.e. union is taken over categories).

In [ ]:
head $DATA/regenie_WES_mask_file.txt | column -t

**Question**
 - The mask defines which variant classes go into a test. What is the trade-off between a strict mask (only predicted loss-of-function) and a loose one?

### Checking input files

To assess the concordance between the input files for building masks, we can use **--check-burden-files** which will generate a report in **file_masks_report.txt** containing:

- for each set, the list the variants in the set-list file which are unrecognized (not genotyped or not present in annotation file for the set)

- for each mask, the list of annotations in the mask definition file which are not in the annotation file

Additionally, we can use **--strict-check-burden** to enforce full agreement between the three files (if not, program will terminate) :

- all genotyped variants in the set list file must be in the annotation file (for the corresponding set)

- all annotations in the mask definition file must be present in the annotation file

In [ ]:
source $CONDA_ACTIVATE $CONDA_ENV
regenie \
  --step 2 \
  --bed $DATA/ukb23150_c22_2024_WES_1826 \
  --ref-first \
  --chr 22 \
  --phenoFile $DATA/covar_2024_WES_1826_phenotype.txt \
  --bt \
  --strict \
  --firth --approx \
  --bsize 1000 \
  --pred $DATA/regenie_step1_WES_pred.list \
  --check-burden-files \
  --anno-file $DATA/regenie_WES_anno_file.txt \
  --set-list $DATA/regenie_WES_set_list_CHR_POS.txt \
  --mask-def $DATA/regenie_WES_mask_file.txt \
  --skip-test \
  --strict-check-burden \
  --out ./output/burden_check_WES
conda deactivate

**Question**
 - This is the gene-based run. Compare its command with the single-variant one: which options turn it into a gene-level test?

### AAF file

Both functional annotations and alternative allele frequency (AAF) cutoffs are used when building masks (e.g. only considering LoF sites where AAF is below 1%).

By default, the AAF for each variant is computed from the sample but alternatively, the user can specify variant AAFs using this file.

### AAF cutoffs

Option **--aaf-bins** specifies the AAF upper bounds used to generate burden masks (AAF and not MAF [minor allele frequency] is used when deciding which variants go into a mask).

By default, a mask based on singleton sites are always included.

For example, **--aaf-bins 0.01,0.05** will generate 3 burden masks for AAFs in [0,0.01], [0,0.05] and singletons.

**Question**
 - The AAF bins set a frequency ceiling for 'rare'. Why does including common variants defeat the purpose of the test?

## SKAT/ACAT tests

The option **--vc-tests** is used to specify the gene-based tests to run. By default, these tests use all variants in each mask category.

If you'd like to only include variants whose AAF is below a given threshold ,e.g. only including rare variants, you can use --vc-maxAAF.

For example, **--vc-tests skato,acato-full** will run SKATO and ACATO (both using the default grid of 8 rho values for the SKATO models) and the p-values for SKAT, SKATO, ACATV and ACATO will be output.

**Ultra-rare variants** (defined by default as MAC ≤ 10, see --vc-MACthr) are collapsed into a burden mask which is then included in the tests instead of the individual variants.

## Joint test for burden masks

The ACAT test combines the p-values of the individual burden masks using the Cauchy combination method.

If you only want to output the results for the joint tests (ignore the marginal tests), use **--joint-only**.

In [ ]:
source $CONDA_ACTIVATE $CONDA_ENV
for chr in {1..22}; do
    regenie \
      --step 2 \
      --bed $DATA/ukb23150_c${chr}_2024_WES_1826 \
      --chr ${chr} \
      --ref-first \
      --phenoFile $DATA/covar_2024_WES_1826_phenotype.txt \
      --bt \
      --strict \
      --firth --approx \
      --bsize 1000 \
      --pred $DATA/regenie_step1_WES_pred.list \
      --check-burden-files \
      --anno-file $DATA/regenie_WES_anno_file_chr${chr}.txt \
      --set-list $DATA/regenie_WES_set_list_chr${chr}_CHR_POS.txt \
      --mask-def $DATA/regenie_WES_mask_file.txt \
      --rgc-gene-p \
      --vc-tests skato,acato-full \
      --joint acat,sbat \
      --vc-MACthr 10 \
      --out ./output/WES_gene_based_testing_chr${chr}
done
conda deactivate

**Questions**
 - SKAT and ACAT are both run. SKAT is a variance-component test and ACAT combines p-values — when would each be the better choice?
 - A burden test assumes all rare variants in a gene act in the same direction. Why is that often wrong?

For each set, this will produce masks using 3 AAF cutoffs (singletons, 5% and 10% AAF).

The masks are written to PLINK bed file (in **_masks.{bed,bim,fam}**) and tested for association with each trait (summary stats in **_phenotype_name.regenie**).

Additionally, a header line is included (starting with ##) which contains mask definition information.

Masks will have name set_name.mask_name.AAF_cutoff with the chromosome and physical position having been defined in the set list file, and the reference allele being ref, and the alternate allele corresponding to **mask_name.AAF_cutoff**.

When using **--rgc-gene-p**, it will apply the single p-value per gene GENE_P strategy using all masks.

**Question**
 - The AAF bins set a frequency ceiling for 'rare'. Why does including common variants defeat the purpose of the test?

### View the result of REGENIE-GWAS


In [ ]:
# combined the regenie results by chromosomes
awk 'FNR <= 2 && NR > 2 { next } { print }' \
    ./output/WES_gene_based_testing_chr*_phenotype.regenie \
    > ./output/WES_gene_based_testing_all.regenie

In [ ]:
head ./output/WES_gene_based_testing_all.regenie | column -t

**Question**
 - The results are now one row per gene and mask, not per variant. How many tests were run, and what significance threshold does that call for?

In [ ]:
# search for the most significant gene
awk 'NR>1 {print $1,$2,$3,$12}' \
  ./output/WES_gene_based_testing_all.regenie | sort -k4,4gr 2>/dev/null | head -1 || true

**Question**
 - What is the most significant gene? Does it have a known connection to Charcot-Marie-Tooth disease?

In [ ]:
# Prepare for plotting the results
regenie_results= data.table::fread("./output/WES_gene_based_testing_all.regenie")
regenie_results[, P := 10^(-LOG10P)] %>%
    data.table::setnames(
        c("CHROM", "GENPOS", "ID"),
        c("CHR", "BP", "SNP")
    )
regenie_results %>%
    data.table::fwrite("./output/WES_gene_based_testing_all_v2.regenie")

In [ ]:
# running about 0.5 minutes
# Note: the alpha used here is 5e-8, but should be set as 0.05/N_genes!
options(repr.plot.width = 16,
        repr.plot.height = 4,
        repr.plot.res = 600)
source("$DATA/plotPlink.R")

# plot the results (.regenie is the output file from regenie)
plots= plot_qqman(
  plink_assoc_file= "./output/WES_gene_based_testing_all_v2.regenie",
  pheno_name= "CMT",
  save_plot = FALSE,
  lambda1_qq_pos = c(1.48, -5.5),
  lambda2_qq_pos = c(1.1, -4)
)
print(plots$manhattan_plot)

**Question**
 - The note says alpha 5e-8 is used but should be set by the number of genes. What would be a defensible threshold here?

In [ ]:
# running about 0.5 minutes
options(repr.plot.width = 4.1,
        repr.plot.height = 4.1,
        repr.plot.res = 600)
print(plots$qq_plot)

## Excercise F: Calculate GWAS power


Now we generate a statistical power analysis plot for GWAS studies.

Supports binary (case-control) traits over a range of odds ratios and minor allele frequencies, and quantitative traits over a range of effect sizes and minor allele frequencies.


### Part1 : Quantitative traits

In our above example, standing height is the quantitative trait.

In [ ]:
# calculate the standard deviation of the quantitative trait
import pandas as pd
df = pd.read_table(
    '$DATA/European_1w_phenotypes.txt',
    sep = '\\s+',
    header = 0
)
print(df.shape)
df['Standing_height'].std(skipna=True)

**Question**
 - Power depends on effect size, allele frequency and sample size. Which of the three is easiest to change in a real study, and which is fixed by biology?

In [ ]:
options(repr.plot.width = 16,
        repr.plot.height = 8,
        repr.plot.res = 600)
source("$DATA/plotPlink.R")
power_results_qt <- plot_gwas_power(
        trait_type = "qt",
        sd_trait = 0.09365788681305078,
        N = 10000,
        maf_levels = c(0.01, 0.02, 0.05, 0.10, 0.20, 0.50),
        effect_size = seq(0.01, 0.10, 0.001),
        save_plot = FALSE
    )
print(power_results_qt$plot)

**Question**
 - At what sample size does the power curve reach 80% for a realistic effect size?

### Part2: Binary traits

Now we generate a statistical power analysis plot for a given range of odds ratios and minor allele frequencies in a **case-control** GWAS study. 

Statistical power is crucial for designing a successful GWAS.

It helps you determine the probability of detecting a true association, given a specific sample size, allele frequency, and effect size (Odds Ratio).

### Example Usage

Let's run the example with a sample dataset:

- Cases: 4,324

- Controls: 93,945

- Odds Ratios: Ranging from 1.01 to 2.00

- MAF: 0.01, 0.02, 0.05, 0.10, 0.20, 0.50

In [ ]:
options(repr.plot.width = 16,
        repr.plot.height = 8,
        repr.plot.res = 600)
source("$DATA/plotPlink.R")

power_results <- plot_gwas_power(
        trait_type = "bt",
        n_cases = 4324,
        n_controls = 93945,
        maf_levels = c(0.01, 0.02, 0.05, 0.10, 0.20, 0.50),
        or_range = seq(1.01, 2.00, 0.01),
        save_plot = FALSE
    )
print(power_results$plot)

**Question**
 - Compare the binary-trait power curve with the quantitative one. Why does a binary trait need more samples for the same effect?

### Run the cell below to take the quiz

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/gwas/quiz/gene_based_testing.json")
